# Hull Tactical - Market Prediction: Ensembling Trees with Online Training（学習用解説付き写し）

- **コンペ**: [Hull Tactical - Market Prediction](https://www.kaggle.com/competitions/hull-tactical-market-prediction)（匿名化された市場特徴量からS&P500の翌日超過リターンを予測し、0〜2倍のポジションサイズを決定するFeatured Code Competition。賞金$100,000。訓練フェーズは終了しているが、実際の市場データに対して定期的にリーダーボードが更新される「forecasting phase」が2026年6月26日まで継続中）
- **元notebook**: [Hull Tactical-Ensembling-trees-Online-Training-001](https://www.kaggle.com/code/youneseloiarm/hull-tactical-ensembling-trees-online-training-001) by **El Younes**（132 upvotes, Gold, Private Score 0.388）
- **手法の概要**: XGBoost・LightGBM・CatBoostの3つの勾配ブースティング木モデルをアンサンブルし、さらにKaggle評価APIによる**逐次提出のたびにオンライン学習（追加の再学習）を行う**ことで、直近の市場データに追従するモデルを構築する。あわせて、オンライン学習における検証データの取り方（validation_split）に関するよくある誤解を明示的に指摘している教育的な内容も含む。
- **このノートブックについて**: 学習目的の解説付き写しであり、未実行（出力結果は含みません）。コード自体は元notebookの内容をほぼそのまま保持しています。


## 評価指標について

このコンペは、市場平均（S&P500）に対する幾何平均Sharpe比を、ボラティリティ超過（市場の1.2倍を超えるボラティリティ）へのペナルティとリターン未達への二次ペナルティで調整した独自指標（Adjusted Sharpe Ratio）で評価されます。単なるリターンの大きさではなく、**リスク調整後のパフォーマンス**かつ**制約違反への罰則**を組み込んだ設計です。

**なぜこの指標が採用されているか**: 「ポジションサイズ0〜2倍」という裁量を参加者に与えているため、単純にリターン最大化だけを目指すと過度にレバレッジを掛けた高リスク戦略が有利になってしまいます。ボラティリティ制約超過へのペナルティを課すことで、実運用に耐えるリスク管理を伴った戦略を評価できるようにしています。

**このnotebookの手法が指標とどう関わるか**: 直接この指標を損失関数として最適化しているわけではなく、まず翌日リターン（`forward_returns`）を回帰で予測し、その予測値から線形のポジションサイズ変換式（`allocation = 1.0 + 50 * pred`、0〜2にクリップ）で配分を決めています。オンライン学習によって直近の市場regimeに追従することがボラティリティ超過を避ける一因になり得ますが、著者自身がnotebook内で「オンライン学習の検証データの取り方を間違えると意味がなくなる」という重要な注意点を指摘しています（詳細は後述）。

なお、このコンペのCodeタブ上位には同一スコア（17.396等）に集中したnotebookが多数あり、評価指標の境界条件を突いた「degenerate戦略（メトリックハック）」の可能性が高いことが分かっています。今回選んだnotebook（Private Score 0.388）はそのクラスタとは異なる値であり、地に足のついたアプローチと考えられます。


## Introduction（著者による導入）

このチュートリアルnotebookでは、XGBoost・LightGBM・CatBoostのようなアンサンブル機械学習モデルを、**API推論の提出ステップ中にオンライン学習する**方法を扱います。オンライン学習中の1行あたりの再学習は約27秒かかりますが、1年分（252取引日）の予測でも最大2時間程度で収まるため、タイムアウトを過度に心配する必要はない、と著者は述べています。このnotebookはリーク（未来の情報の混入）が無いように設計されています。


In [ ]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
from scipy.stats import pearsonr
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error
import warnings
warnings.filterwarnings("ignore")
import kaggle_evaluation.default_inference_server

**何をしているか（What）**: 3種類の勾配ブースティング木ライブラリ（XGBoost, LightGBM, CatBoost）と、Kaggleのコード提出用評価API（`kaggle_evaluation`）をインポートしています。

**なぜ3つのライブラリを併用するのか（Why）**: それぞれ木の分割方法や正則化の実装が微妙に異なるため、単体モデルより誤差の相関が下がりやすく、アンサンブルによる精度向上が期待できます。


## データ読み込みと概要確認

学習データを読み込み、`train.info()` で列数・欠損状況・型を確認しています。読み込んだデータは98列（`date_id` と D/E/I/M系の匿名化特徴量群、目的変数群）、約9,048行の日次データです。特徴量の一部（`E1`, `E20`, `M1`, `M13`, `M14` など）には数千件規模の欠損があり、後段で `fillna(0)` により補完しています。


In [ ]:
targets = ['forward_returns', 'risk_free_rate', 'market_forward_excess_returns']
features = [ i for i in train.columns if i not in targets]
test_data = train[-180:]
train_data = train[:-180]

In [ ]:
target = "forward_returns"
X_train = train_data[features].fillna(0)
y_train = train_data[target]
X_test = test_data[features].fillna(0)
y_test = test_data[target]

**何をしているか（What）**: 目的変数として `forward_returns`（翌期間のリターン）を選び、直近180日をローカルなテスト検証用に切り出しています。特徴量の欠損値は単純に0で埋めています。

**なぜ末尾180日を切り出すのか（Why）**: 時系列データでは、ランダムにtrain/testを分割すると未来の情報が学習データに混ざる「時間的リーク」が発生します。時系列順にスライスして直近180日をホールドアウトにすることで、実運用に近い「過去のデータで学習し、未来を予測する」評価が可能になります。


## モデル定義とバリデーション付き学習

**何をしているか（What）**: `create_models()` はXGBoost・LightGBM・CatBoostの3モデルをそれぞれ2000本の木、学習率0.01、深さ7というハイパーパラメータで初期化します。`train_with_validation()` はモデルごとに `X_train`/`y_train` を `validation_split` の比率で分割し、検証セットでのMSE/RMSE/MAEを記録しながら学習します。


In [ ]:
def create_models():
    """Create and return the gradient boosting models with the specified parameters."""
    xgb_model = XGBRegressor(
        n_estimators=2000,      # more boosting rounds
        max_depth=7,            # deeper trees
        learning_rate=0.01,     # smaller LR for stability
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=1,
        reg_lambda=1,
        random_state=42,
        tree_method="hist",
        device="cuda"           # remove if no GPU
    )

    lgbm_model = LGBMRegressor(
        n_estimators=2000,
        max_depth=7,
        learning_rate=0.01,
        num_leaves=256,         # more leaves for depth
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=1,
        reg_lambda=1,
        random_state=42,
        device="gpu",           # remove if no GPU
        verbose=-1
    )

    catboost_model = CatBoostRegressor(
        iterations=2000,
        depth=7,
        learning_rate=0.01,
        l2_leaf_reg=3,
        random_seed=42,
        loss_function='RMSE',
        task_type="GPU",        # remove if no GPU
        verbose=False
    )

    return [xgb_model, lgbm_model, catboost_model]

def train_with_validation(models, X_train, y_train, validation_split=0.25):
    """
    Train models with validation on the last portion of training data.

    Parameters
    ----------
    models : list
        List of model objects to train
    X_train : array-like
        Training features
    ...(docstring省略)...
    """
    # ...(トレーニングデータを時系列順にsplitし、各モデルをfitして検証指標を算出する処理)...
    for i, model in enumerate(models):
        model_name = model_names[i]
        print(f"\nTraining {model_name}...")

        # Train the model
        if model_name == 'CatBoost':
            model.fit(X_train_split, y_train_split,
                      eval_set=(X_val, y_val),
                      verbose=False)
        elif model_name == 'LightGBM':
            model.fit(X_train_split, y_train_split,
                      eval_set=(X_val, y_val))
        else:
            model.fit(X_train_split, y_train_split,
                      eval_set=[(X_val, y_val)],
                      verbose=False)

        # Make predictions on validation set
        y_pred = model.predict(X_val)

        # Calculate validation metrics
        mse = mean_squared_error(y_val, y_pred)
        rmse = np.sqrt(mse)
        mae = mean_absolute_error(y_val, y_pred)

        validation_metrics[model_name] = {
            'MSE': mse,
            'RMSE': rmse,
            'MAE': mae
        }

        print(f"{model_name} Validation Metrics:")
        print(f"  MSE: {mse:.4f}, RMSE: {rmse:.4f}, MAE: {mae:.4f}")

        trained_models.append(model)

    return trained_models, validation_metrics

def online_training_pipeline(X_train, y_train, validation_split=0.25):
    """Complete pipeline for creating and training models with validation."""
    print("Creating models...")
    models = create_models()

    print(f"Starting training with validation on last {validation_split*100}% of data...")
    trained_models, validation_metrics = train_with_validation(
        models, X_train, y_train, validation_split
    )

    print(f"\nTraining completed. {len(trained_models)} models trained and validated.")
    return trained_models, validation_metrics

**なぜGPUデバイス指定（`device="cuda"` 等）をコード内に直書きしているのか（Why）**: 木の本数が2000本と多いため、CPUだけでは学習に時間がかかります。特にこのnotebookは後述のオンライン学習で「提出APIから新しい行が来るたびに全モデルを再学習する」ため、1回の学習を高速化することが全体の実行時間に直結します。


## ⚠️ オンライン学習における validation_split の落とし穴（著者による重要な指摘）

このnotebookで最も学びが大きいのが、この節の警告です。

オンライン学習（逐次学習）では、モデルを過去の全履歴データで学習しつつ、検証は**直近の観測値だけ**に限定するのが本来の考え方です。しかし多くの人が誤って次のように設定してしまいます。

```python
validation_split = 0.25
```

もし9,000日分のデータがあるなら、`0.25` は2,250日（約9年分）ものデータを検証セットに割り当てることを意味します。**これはオンライン学習において意味を成しません**。なぜなら：

- オンライン学習は直近の市場regime（値動きの傾向）に追従することが目的なのに、検証を9年分の古いデータに対して行ってしまうと、「直近にどれだけ適合できているか」を全く測れていないことになります。
- 推奨される `validation_split` は、9,000日データセットなら次のようなごく小さい値です。

| 検証したい期間 | 日数 | 9000日に対する割合 | 推奨 validation_split |
|---|---|---|---|
| 30日 | 30 | 30/9000 ≈ 0.0033 | validation_split ≈ 0.003 |
| 22日 | 22 | 22/9000 ≈ 0.00244 | validation_split < 0.0025 |
| 9日 | 9 | 9/9000 ≈ 0.001 | validation_split ≈ 0.001 |
| 1日 | 1 | 1/9000 ≈ 0.00011 | validation_split ≈ 0.0001 |

**✅ 要点**: オンライン学習を使うなら、検証期間は必ず「直近かつごく短い」期間にすること。9年分もの古いデータで検証させてしまうと、モデルがどれだけ最新の市場regimeに適応できているかを測れなくなります。目安として `0.0001 〜 0.003` の範囲を検討すべきです（検証したい直近日数に応じて調整）。

**なぜこれが重要な学びか（Why）**: 「validation_splitを大きくすれば安心」という一般的な機械学習の直感が、時系列のオンライン学習では逆効果になりうる好例です。データの独立同分布（i.i.d.）を仮定した通常の検証手法を、時間的な構造を持つデータにそのまま適用すると誤った結論に至る、という典型的な落とし穴を明示的に指摘している点が教育的価値の高い部分です。


## 評価用スコアリング関数（Adjusted Sharpe Ratio）

コンペの評価指標そのもの（市場平均超過に対するボラティリティペナルティ・リターン未達ペナルティ付きの調整済みSharpe比）を、ローカルでシミュレーションするための実装です。


In [ ]:
market_excess_cumulative = (1 + market_excess_returns).prod()
market_mean_excess_return = (market_excess_cumulative) ** (1 / len(solution)) - 1
market_std = solution['forward_returns'].std()

market_volatility = float(market_std * np.sqrt(trading_days_per_yr) * 100)

# Calculate the volatility penalty
excess_vol = max(0, strategy_volatility / market_volatility - 1.2) if market_volatility > 0 else 0
vol_penalty = 1 + excess_vol

# Calculate the return penalty
return_gap = max(
    0,
    (market_mean_excess_return - strategy_mean_excess_return) * 100 * trading_days_per_yr,
)
return_penalty = 1 + (return_gap**2) / 100

# Adjust the Sharpe ratio by the volatility and return penalty
adjusted_sharpe = sharpe / (vol_penalty * return_penalty)
return min(float(adjusted_sharpe), 1_000_000)

**何をしているか（What）**: 戦略のボラティリティが市場の1.2倍を超えた分（`excess_vol`）に応じてペナルティを掛け算し（`vol_penalty`）、さらに市場平均に対してリターンが不足した分の二乗（`return_gap**2`）にもペナルティを掛けます（`return_penalty`）。最終的なシャープ比は両ペナルティで割り引かれます。

**なぜ二乗ペナルティなのか（Why）**: リターン不足に対して単純な比例ペナルティではなく二乗にすることで、「少し届かない」場合よりも「大きく届かない」場合をより強く罰する設計になっています。これにより、極端に保守的すぎる（リターンをほとんど取りに行かない）戦略も評価上不利になるよう誘導されています。


In [ ]:
preds = ensemble_predict(trained_models, X_test)

In [ ]:
### Simulation:
solution = train[["date_id","forward_returns","risk_free_rate"]][-180:]
submission = pd.DataFrame()
submission = train[["date_id"]][-180:]
submission["prediction"] = preds
submission.columns = ["date_id","prediction"]
# Turn negatives into 0, keep positives as they are
submission["prediction"] = submission["prediction"].apply(lambda x: x*2 if x > 0 else 0)#0.0089)

# Run scoring
score_value = score(solution, submission, row_id_column_name="date_id")
print("Adjusted Sharpe Score:", score_value)

# Adjusted Sharpe Score: 0.045540865969479316

In [ ]:
allocation = 1.0 + 50 * preds
allocation = np.clip(allocation, 0.0, 2.0)
# Turn negatives into 0, keep positives as they are
submission["prediction"] = allocation

# Run scoring
score_value = score(solution, submission, row_id_column_name="date_id")
print("Adjusted Sharpe Score:", score_value)

# Adjusted Sharpe Score: 1.066404002532888

**何をしているか（What）**: 2種類のポジションサイズ変換式を試しています。1つ目は予測値を単純に2倍して負値を切り捨てる方式（Adjusted Sharpe Score ≈ 0.046、あまり良くない）。2つ目は `allocation = 1.0 + 50 * preds` として予測値を50倍に増幅した上で1.0を中心に0〜2の範囲にクリップする方式で、こちらはスコアが 1.066 まで大きく改善しています。

**なぜ増幅係数（×50）が効くのか（Why）**: 予測されるリターン（`forward_returns`）は日次でごく小さい値（数%未満）のため、そのままではポジションサイズの変化がほとんど生まれません。予測値を大きく増幅してから0〜2の範囲にクリップすることで、「強気の予測が出たら積極的にポジションを取り、弱気なら守りに入る」という意味のある配分変化を作り出しています。ただしこの倍率（50）は経験的に調整されたハイパーパラメータであり、根拠となる理論的な導出過程はnotebook内には示されていません。


## リーク防止のためのラグ特徴量作成

**何をしているか（What）**: 目的変数（`forward_returns`, `risk_free_rate`, `market_forward_excess_returns`）を1期シフト（`shift(1)`）した「ラグ付き目的変数」を新しい特徴量として追加しています。


In [ ]:
def create_lagged_features(dataframe):
    """Add lagged target variables to the dataset."""
    target_columns = ['forward_returns', 'risk_free_rate', 'market_forward_excess_returns']
    dataframe["is_scored"] = False

    for target_col in target_columns:
        dataframe[f"lagged_{target_col}"] = dataframe[target_col].shift(1)

    return dataframe

train = create_lagged_features(train)

**なぜこれがリーク防止になるのか（Why）**: Hull Tacticalの実運用シナリオでは、ある日の取引を決める時点でその日の`forward_returns`（未来のリターン）は当然分かりません。しかし前日までのリターンの推移（ラグ特徴量）は既知の情報として利用可能です。目的変数そのものをそのまま特徴量に使うと明確なリーク（カンニング）になりますが、1期シフトすることで「過去の実績を参考にする」という現実的な情報の使い方に変換しています。


## オンライン学習を組み込んだ推論用 `predict()` 関数

ここが本notebookの核心部分です。Kaggleの評価API（`kaggle_evaluation.default_inference_server`）は、日ごとに新しい特徴量の行を1つずつ `predict()` 関数に渡してきます。この関数は、**単に学習済みモデルで予測するだけでなく、新しい行が来るたびにその時点までのデータでモデルを再学習してから予測する**、という逐次オンライン学習を実装しています。


In [ ]:
import time
count = 0
online_X_train = None
online_y_train = None

def predict(test_df: "pl.DataFrame") -> float:
    global features, train, count, online_X_train, online_y_train

    start_time = time.time()  # Start timing
    test_df = test_df.to_pandas()
    Date_id = test_df["date_id"].unique()[0]
    if count == 0:
        # Initialize training set up to the current date
        train_df = train.loc[train["date_id"] < Date_id]
        X_train, y_train = train_df[features].fillna(0), train_df[target].fillna(0)

        # Example usage: training all models
        trained_models, metrics = online_training_pipeline(X_train, y_train, validation_split = 0.0025)

        # Initialize online training data
        online_X_train, online_y_train = X_train.copy(), y_train.copy()
    else:
        # ...(count > 0 のとき: 前回までのオンライン学習データに
        #      新しい1行を追加して再学習し、直近のごく短いwindowで検証する処理が続く)...
        pass

    # (推論: trained_models を使って ensemble_predict → allocation 変換 → return)

    # Measure elapsed time
    elapsed_time = time.time() - start_time
    print(f"Step {count} (FULL PROCESSING - last is_scored=True): {elapsed_time:.4f} seconds")
    print("="*25)
    count += 1
    return float(allocation)

**何をしているか（What）**: `count == 0`（最初の呼び出し）の場合、その日付より前のすべての履歴データでモデルを新規学習します。このとき `validation_split=0.0025` と、まさに上で解説した「オンライン学習には極小のvalidation_splitを使うべき」という教訓が実際に適用されています。以降の呼び出し（`count > 0`）では、新しく届いた1行を学習データに追加し、モデルを再学習してから予測を行います（オンライン/逐次学習）。

**なぜ毎回再学習するのか（Why）**: 金融市場は非定常（statistically non-stationary）で、過去のパターンが将来も同じように成り立つとは限りません。1回学習したモデルを固定して使い続けるのではなく、新しい市場データが手に入るたびにモデルを更新することで、直近の市場regimeへの適応力を保とうとしています。ただしこのアプローチは計算コストが高く（1行あたり約27秒）、著者自身も「1年分で最大2時間程度」と実行時間について言及しています。


In [ ]:
import os
inference_server = kaggle_evaluation.default_inference_server.DefaultInferenceServer(predict)

if __name__ == "__main__":
    if os.getenv("KAGGLE_IS_COMPETITION_RERUN"):
        inference_server.serve()
    else:
        inference_server.run_local_gateway(("/kaggle/input/hull-tactical-market-prediction/",))

**何をしているか（What）**: Kaggleの評価API用サーバーを起動しています。本番の再実行環境（`KAGGLE_IS_COMPETITION_RERUN`環境変数がセットされている場合）では `serve()` で実際の評価ループに接続し、ローカル開発時は `run_local_gateway()` でサンプルデータを使ったシミュレーションを行います。

**実行ログの例（著者の出力より）**:
```
Training samples: 8957, Validation samples: 23
Training XGBoost...
XGBoost Validation Metrics: MSE: 0.0001, RMSE: 0.0071, MAE: 0.0053
...
Training completed. 3 models trained and validated.
Step 0 (FULL PROCESSING - last is_scored=True): 38.3290 seconds
=========================
```
このログから、`validation_split=0.0025` のもとで約9,000件のうちわずか23件（≈0.25%）だけが検証に使われていることが確認でき、上の警告セクションの推奨レンジ通りに設定されていることが分かります。


## この手法から学べる主要テクニック

- 複数の勾配ブースティング木ライブラリ（XGBoost/LightGBM/CatBoost）を組み合わせたアンサンブル。
- **Kaggle評価APIの推論ステップ内でオンライン学習（逐次再学習）を行う**、時系列コンペティション特有の実装パターン。
- オンライン学習における検証データの取り方（validation_splitは直近の極小ウィンドウにすべき）という、時系列データ特有の落とし穴の明示的な指摘。
- 目的変数のラグ特徴量化による、時間的リークを避けたシグナル活用。
- 予測値をそのまま使わず、増幅・クリップを組み合わせてポジションサイズに変換する後処理設計（ただし倍率の根拠は経験則）。
- コンペ独自の評価指標（ボラティリティ・リターン未達ペナルティ付きSharpe比）をローカルで再現し、後処理のパラメータ（増幅係数など）を評価指標に対して直接チューニングする方法。
